
# Nova seção

In [47]:
# ==========================================
# PROJETO INTEGRADOR - HITECH
# Sistema de Gestão de Desenvolvimento
# ==========================================

print("====================================")
print("       SISTEMA HITECH")
print("====================================")

print("Ambiente de desenvolvimento iniciado!")
print("Projeto: Modelagem de Banco de Dados")
print("SGBD previsto: MySQL 8.0")
print("Linguagem: Python")

       SISTEMA HITECH
Ambiente de desenvolvimento iniciado!
Projeto: Modelagem de Banco de Dados
SGBD previsto: MySQL 8.0
Linguagem: Python


In [48]:
!pip install mysql-connector-python pandas sqlalchemy

1. CONFIGURAÇÃO DO AMBIENTE

2. CONEXÃO COM O BANCO DE DADOS

3. CRIAÇÃO DO BANCO DE DADOS

4. CRIAÇÃO DAS TABELAS

5. REGRAS DE NEGÓCIO E TRIGGERS

6. INSERÇÃO DE DADOS

7. OPERAÇÕES DO SISTEMA

8. INTERFACE GRÁFICA

9. TESTES E VALIDAÇÕES

In [49]:



!apt-get update -qq
!apt-get install -y mysql-server

print("Instalação do MySQL concluída!")


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
mysql-server is already the newest version (8.0.46-0ubuntu0.24.04.4).
0 upgraded, 0 newly installed, 0 to remove and 31 not upgraded.
Instalação do MySQL concluída!


In [50]:

# INICIAR O SERVIDOR MYSQL

!service mysql start

print("Servidor MySQL iniciado!")


[ OK ]
Servidor MySQL iniciado!


In [51]:

# VERIFICAR O MYSQL

!mysql --version
!sudo mysql -e "SELECT VERSION();"


mysql  Ver 8.0.46-0ubuntu0.24.04.4 for Linux on x86_64 ((Ubuntu))
+-------------------------+
| VERSION()               |
+-------------------------+
| 8.0.46-0ubuntu0.24.04.4 |
+-------------------------+


# criação do banco de dados

In [52]:

# CRIAR O BANCO DE DADOS HITECH

!sudo mysql -e "CREATE DATABASE IF NOT EXISTS hitech_erp CHARACTER SET utf8mb4 COLLATE utf8mb4_unicode_ci;"

# CRIAR USUÁRIO PARA O NOSSO SISTEMA
!sudo mysql -e "CREATE USER IF NOT EXISTS 'hitech_user'@'127.0.0.1' IDENTIFIED BY 'Hitech@2026';"

# DAR PERMISSÕES AO USUÁRIO
!sudo mysql -e "GRANT ALL PRIVILEGES ON hitech_erp.* TO 'hitech_user'@'127.0.0.1';"

!sudo mysql -e "FLUSH PRIVILEGES;"

print("Banco de dados Hitech criado com sucesso!")


Banco de dados Hitech criado com sucesso!


In [53]:

!sudo mysql -e "SHOW DATABASES;"


+--------------------+
| Database           |
+--------------------+
| hitech_erp         |
| information_schema |
| mysql              |
| performance_schema |
| sys                |
+--------------------+


In [54]:

import mysql.connector

conexao = mysql.connector.connect(
    host="127.0.0.1",
    user="hitech_user",
    password="Hitech@2026",
    database="hitech_erp"
)

if conexao.is_connected():
    print("Conexão com o banco Hitech realizada com sucesso!")

conexao.close()


Conexão com o banco Hitech realizada com sucesso!


# criação das tabelas (7)

In [55]:

import mysql.connector

conexao = mysql.connector.connect(
    host="127.0.0.1",
    user="hitech_user",
    password="Hitech@2026",
    database="hitech_erp"
)

cursor = conexao.cursor()

# 1. TABELA CLIENTE
cursor.execute("""
CREATE TABLE IF NOT EXISTS cliente (
    id_cliente INT AUTO_INCREMENT PRIMARY KEY,
    nome VARCHAR(120) NOT NULL,
    email VARCHAR(120) UNIQUE,
    telefone VARCHAR(20),
    data_cadastro DATE NOT NULL,
    ativo BOOLEAN NOT NULL DEFAULT TRUE
) ENGINE=InnoDB;
""")

# 2. TABELA NIVEL_PROFISSIONAL
cursor.execute("""
CREATE TABLE IF NOT EXISTS nivel_profissional (
    id_nivel INT AUTO_INCREMENT PRIMARY KEY,
    nome_nivel VARCHAR(20) NOT NULL UNIQUE,
    descricao VARCHAR(200),
    complexidade_permitida VARCHAR(20) NOT NULL,
    CHECK (nome_nivel IN ('Junior', 'Pleno', 'Senior')),
    CHECK (complexidade_permitida IN ('Simples', 'Media', 'Alta'))
) ENGINE=InnoDB;
""")

# 3. TABELA FUNCIONARIO
cursor.execute("""
CREATE TABLE IF NOT EXISTS funcionario (
    id_funcionario INT AUTO_INCREMENT PRIMARY KEY,
    id_nivel INT NOT NULL,
    nome VARCHAR(120) NOT NULL,
    email VARCHAR(120) NOT NULL UNIQUE,
    data_admissao DATE NOT NULL,
    ativo BOOLEAN NOT NULL DEFAULT TRUE,

    CONSTRAINT fk_funcionario_nivel
    FOREIGN KEY (id_nivel)
    REFERENCES nivel_profissional(id_nivel)
    ON DELETE RESTRICT
    ON UPDATE CASCADE
) ENGINE=InnoDB;
""")

conexao.commit()

print("As três primeiras tabelas foram criadas com sucesso!")

cursor.close()
conexao.close()


As três primeiras tabelas foram criadas com sucesso!


In [56]:

conexao = mysql.connector.connect(
    host="127.0.0.1",
    user="hitech_user",
    password="Hitech@2026",
    database="hitech_erp"
)

cursor = conexao.cursor()

cursor.execute("SHOW TABLES")

print("TABELAS EXISTENTES NO BANCO HITECH:\n")

for tabela in cursor.fetchall():
    print("-", tabela[0])

cursor.close()
conexao.close()


TABELAS EXISTENTES NO BANCO HITECH:

- alocacao_tarefa
- cliente
- funcionario
- historico_tarefa
- nivel_profissional
- projeto
- tarefa


In [57]:

import mysql.connector

conexao = mysql.connector.connect(
    host="127.0.0.1",
    user="hitech_user",
    password="Hitech@2026",
    database="hitech_erp"
)

cursor = conexao.cursor()

# 4. TABELA PROJETO
cursor.execute("""
CREATE TABLE IF NOT EXISTS projeto (
    id_projeto INT AUTO_INCREMENT PRIMARY KEY,
    id_cliente INT NOT NULL,
    nome VARCHAR(150) NOT NULL,
    descricao TEXT,
    data_inicio DATE NOT NULL,
    data_prevista_fim DATE,
    data_fim DATE,
    status VARCHAR(30) NOT NULL DEFAULT 'Planejado',

    CONSTRAINT hitech_fk_projeto_cliente
    FOREIGN KEY (id_cliente)
    REFERENCES cliente(id_cliente)
    ON DELETE RESTRICT
    ON UPDATE CASCADE
) ENGINE=InnoDB;
""")

# 5. TABELA TAREFA
cursor.execute("""
CREATE TABLE IF NOT EXISTS tarefa (
    id_tarefa INT AUTO_INCREMENT PRIMARY KEY,
    id_projeto INT NOT NULL,
    titulo VARCHAR(150) NOT NULL,
    descricao TEXT,
    complexidade VARCHAR(20) NOT NULL,
    status VARCHAR(30) NOT NULL DEFAULT 'Pendente',
    data_criacao DATE NOT NULL,
    prazo DATE,

    CONSTRAINT hitech_fk_tarefa_projeto
    FOREIGN KEY (id_projeto)
    REFERENCES projeto(id_projeto)
    ON DELETE RESTRICT
    ON UPDATE CASCADE,

    CHECK (complexidade IN ('Simples', 'Media', 'Alta'))
) ENGINE=InnoDB;
""")

# 6. TABELA ALOCACAO_TAREFA
cursor.execute("""
CREATE TABLE IF NOT EXISTS alocacao_tarefa (
    id_alocacao INT AUTO_INCREMENT PRIMARY KEY,
    id_tarefa INT NOT NULL,
    id_funcionario INT NOT NULL,
    data_atribuicao DATETIME NOT NULL DEFAULT CURRENT_TIMESTAMP,
    data_fim DATETIME DEFAULT NULL,
    ativa BOOLEAN NOT NULL DEFAULT TRUE,

    CONSTRAINT hitech_fk_alocacao_tarefa
    FOREIGN KEY (id_tarefa)
    REFERENCES tarefa(id_tarefa)
    ON DELETE RESTRICT
    ON UPDATE CASCADE,

    CONSTRAINT hitech_fk_alocacao_funcionario
    FOREIGN KEY (id_funcionario)
    REFERENCES funcionario(id_funcionario)
    ON DELETE RESTRICT
    ON UPDATE CASCADE
) ENGINE=InnoDB;
""")

# 7. TABELA HISTORICO_TAREFA
cursor.execute("""
CREATE TABLE IF NOT EXISTS historico_tarefa (
    id_historico INT AUTO_INCREMENT PRIMARY KEY,
    id_tarefa INT NOT NULL,
    id_funcionario INT,
    tipo_evento VARCHAR(50) NOT NULL,
    status_anterior VARCHAR(30),
    status_novo VARCHAR(30),
    descricao TEXT,
    data_evento DATETIME NOT NULL DEFAULT CURRENT_TIMESTAMP,

    CONSTRAINT hitech_fk_historico_tarefa
    FOREIGN KEY (id_tarefa)
    REFERENCES tarefa(id_tarefa)
    ON DELETE RESTRICT
    ON UPDATE CASCADE,

    CONSTRAINT hitech_fk_historico_funcionario
    FOREIGN KEY (id_funcionario)
    REFERENCES funcionario(id_funcionario)
    ON DELETE SET NULL
    ON UPDATE CASCADE
) ENGINE=InnoDB;
""")

conexao.commit()

print("As quatro tabelas foram criadas ou já existiam.")

cursor.execute("SHOW TABLES")

print("\nTABELAS DO BANCO HITECH:")
tabelas = cursor.fetchall()

for tabela in tabelas:
    print("✔", tabela[0])

print("\nTotal de tabelas:", len(tabelas))

cursor.close()
conexao.close()


As quatro tabelas foram criadas ou já existiam.

TABELAS DO BANCO HITECH:
✔ alocacao_tarefa
✔ cliente
✔ funcionario
✔ historico_tarefa
✔ nivel_profissional
✔ projeto
✔ tarefa

Total de tabelas: 7


In [58]:

import mysql.connector

conexao = mysql.connector.connect(
    host="127.0.0.1",
    user="hitech_user",
    password="Hitech@2026",
    database="hitech_erp"
)

cursor = conexao.cursor()

cursor.execute("""
SELECT
    TABLE_NAME,
    COLUMN_NAME,
    REFERENCED_TABLE_NAME,
    REFERENCED_COLUMN_NAME
FROM information_schema.KEY_COLUMN_USAGE
WHERE TABLE_SCHEMA = 'hitech_erp'
AND REFERENCED_TABLE_NAME IS NOT NULL
ORDER BY TABLE_NAME;
""")

print("RELACIONAMENTOS DO BANCO HITECH")
print("--------------------------------")

for linha in cursor.fetchall():
    print(
        f"{linha[0]}.{linha[1]} -> "
        f"{linha[2]}.{linha[3]}"
    )

cursor.close()
conexao.close()


RELACIONAMENTOS DO BANCO HITECH
--------------------------------
alocacao_tarefa.id_funcionario -> funcionario.id_funcionario
alocacao_tarefa.id_tarefa -> tarefa.id_tarefa
funcionario.id_nivel -> nivel_profissional.id_nivel
historico_tarefa.id_funcionario -> funcionario.id_funcionario
historico_tarefa.id_tarefa -> tarefa.id_tarefa
projeto.id_cliente -> cliente.id_cliente
tarefa.id_projeto -> projeto.id_projeto


In [59]:

conexao = mysql.connector.connect(
    host="127.0.0.1",
    user="hitech_user",
    password="Hitech@2026",
    database="hitech_erp"
)

cursor = conexao.cursor()

cursor.execute("SHOW TABLES")

print("BANCO HITECH")
print("============")

for tabela in cursor.fetchall():
    print("✔", tabela[0])

cursor.execute("""
SELECT COUNT(*)
FROM information_schema.TABLES
WHERE TABLE_SCHEMA = 'hitech_erp'
""")

print("\nTotal de tabelas:", cursor.fetchone()[0])

cursor.close()
conexao.close()


BANCO HITECH
✔ alocacao_tarefa
✔ cliente
✔ funcionario
✔ historico_tarefa
✔ nivel_profissional
✔ projeto
✔ tarefa

Total de tabelas: 7


In [60]:

conexao = mysql.connector.connect(
    host="127.0.0.1",
    user="hitech_user",
    password="Hitech@2026",
    database="hitech_erp"
)

cursor = conexao.cursor()

cursor.execute("""
INSERT INTO nivel_profissional
(nome_nivel, descricao, complexidade_permitida)
VALUES
('Junior', 'Desenvolvedor iniciante', 'Simples'),
('Pleno', 'Desenvolvedor intermediario', 'Media'),
('Senior', 'Desenvolvedor experiente', 'Alta')
""")

conexao.commit()

print("Níveis profissionais cadastrados!")

cursor.execute("SELECT * FROM nivel_profissional")

for linha in cursor.fetchall():
    print(linha)

cursor.close()
conexao.close()


IntegrityError: 1062 (23000): Duplicate entry 'Junior' for key 'nivel_profissional.nome_nivel'

In [ ]:

conexao = mysql.connector.connect(
    host="127.0.0.1",
    user="hitech_user",
    password="Hitech@2026",
    database="hitech_erp"
)

cursor = conexao.cursor()

cursor.executemany("""
INSERT INTO cliente
(nome, email, telefone, data_cadastro, ativo)
VALUES (%s, %s, %s, CURDATE(), TRUE)
""", [
    ("Ana Souza", "ana@exemplo.com", "11999990001"),
    ("Carlos Lima", "carlos@exemplo.com", "11999990002"),
    ("Marina Alves", "marina@exemplo.com", "11999990003"),
    ("Pedro Santos", "pedro@exemplo.com", "11999990004"),
    ("Juliana Rocha", "juliana@exemplo.com", "11999990005")
])

conexao.commit()

print("Clientes cadastrados!")

cursor.close()
conexao.close()


In [ ]:

conexao = mysql.connector.connect(
    host="127.0.0.1",
    user="hitech_user",
    password="Hitech@2026",
    database="hitech_erp"
)

cursor = conexao.cursor()

cursor.executemany("""
INSERT INTO funcionario
(id_nivel, nome, email, data_admissao, ativo)
VALUES (%s, %s, %s, CURDATE(), TRUE)
""", [
    (1, "Lucas Mendes", "lucas@hitech.com"),
    (1, "Bruna Costa", "bruna@hitech.com"),
    (2, "Rafael Martins", "rafael@hitech.com"),
    (2, "Camila Ferreira", "camila@hitech.com"),
    (3, "Gabriel Oliveira", "gabriel@hitech.com")
])

conexao.commit()

print("Funcionários cadastrados!")

cursor.close()
conexao.close()


In [ ]:

conexao = mysql.connector.connect(
    host="127.0.0.1",
    user="hitech_user",
    password="Hitech@2026",
    database="hitech_erp"
)

cursor = conexao.cursor()

cursor.executemany("""
INSERT INTO projeto
(id_cliente, nome, descricao, data_inicio, data_prevista_fim, status)
VALUES (%s, %s, %s, CURDATE(), DATE_ADD(CURDATE(), INTERVAL 60 DAY), 'Em andamento')
""", [
    (1, "Sistema de Vendas", "Sistema de controle comercial"),
    (2, "Sistema Financeiro", "Gestao financeira empresarial"),
    (3, "Portal Corporativo", "Portal de atendimento"),
    (4, "Sistema de Estoque", "Controle de produtos e estoque"),
    (5, "Aplicativo Empresarial", "Aplicativo de gestao")
])

conexao.commit()

print("Projetos cadastrados!")

cursor.close()
conexao.close()


In [ ]:

conexao = mysql.connector.connect(
    host="127.0.0.1",
    user="hitech_user",
    password="Hitech@2026",
    database="hitech_erp"
)

cursor = conexao.cursor()

cursor.executemany("""
INSERT INTO tarefa
(id_projeto, titulo, descricao, complexidade, status, data_criacao, prazo)
VALUES (%s, %s, %s, %s, 'Pendente', CURDATE(),
        DATE_ADD(CURDATE(), INTERVAL 15 DAY))
""", [
    (1, "Criar tela de login", "Desenvolver autenticacao", "Simples"),
    (1, "Criar cadastro de clientes", "Modulo de clientes", "Media"),
    (2, "Implementar calculos financeiros", "Calculos e regras", "Alta"),
    (3, "Criar pagina inicial", "Pagina principal", "Simples"),
    (4, "Criar controle de estoque", "Gerenciamento de produtos", "Media"),
    (5, "Desenvolver arquitetura do aplicativo", "Arquitetura geral", "Alta")
])

conexao.commit()

print("Tarefas cadastradas!")

cursor.close()
conexao.close()


In [ ]:
# Ativar a permissão de criação de funções/triggers de forma global no MySQL utilizando o root do sistema
!sudo mysql -e "SET GLOBAL log_bin_trust_function_creators = 1;"

import mysql.connector

conexao = mysql.connector.connect(
    host="127.0.0.1",
    user="hitech_user",
    password="Hitech@2026",
    database="hitech_erp"
)

cursor = conexao.cursor()

cursor.execute("DROP TRIGGER IF EXISTS validar_alocacao")

cursor.execute("""
CREATE TRIGGER validar_alocacao
BEFORE INSERT ON alocacao_tarefa
FOR EACH ROW
BEGIN
    DECLARE nivel VARCHAR(20);
    DECLARE complexidade_tarefa VARCHAR(20);

    SELECT np.nome_nivel
    INTO nivel
    FROM funcionario f
    JOIN nivel_profissional np
        ON np.id_nivel = f.id_nivel
    WHERE f.id_funcionario = NEW.id_funcionario
      AND f.ativo = TRUE;

    SELECT complexidade
    INTO complexidade_tarefa
    FROM tarefa
    WHERE id_tarefa = NEW.id_tarefa;

    IF nivel IS NULL THEN
        SIGNAL SQLSTATE '45000'
        SET MESSAGE_TEXT = 'Funcionario inexistente ou inativo';
    END IF;

    IF (nivel = 'Junior' AND complexidade_tarefa <> 'Simples')
       OR (nivel = 'Pleno' AND complexidade_tarefa = 'Alta') THEN
        SIGNAL SQLSTATE '45000'
        SET MESSAGE_TEXT = 'Nivel profissional incompatível com a tarefa';
    END IF;
END
""")

conexao.commit()

print("Trigger de compatibilidade criado!")

cursor.close()
conexao.close()

In [ ]:

conexao = mysql.connector.connect(
    host="127.0.0.1",
    user="hitech_user",
    password="Hitech@2026",
    database="hitech_erp"
)

cursor = conexao.cursor()

cursor.execute("DROP TRIGGER IF EXISTS registrar_nova_tarefa")

cursor.execute("""
CREATE TRIGGER registrar_nova_tarefa
AFTER INSERT ON tarefa
FOR EACH ROW
BEGIN
    INSERT INTO historico_tarefa
    (id_tarefa, tipo_evento, status_novo, descricao)
    VALUES
    (NEW.id_tarefa, 'CRIACAO', NEW.status,
     'Tarefa cadastrada no sistema');
END
""")

conexao.commit()

print("Trigger de histórico criado!")

cursor.close()
conexao.close()


In [ ]:

conexao = mysql.connector.connect(
    host="127.0.0.1",
    user="hitech_user",
    password="Hitech@2026",
    database="hitech_erp"
)

cursor = conexao.cursor()

cursor.execute("DROP TRIGGER IF EXISTS registrar_mudanca_status")

cursor.execute("""
CREATE TRIGGER registrar_mudanca_status
AFTER UPDATE ON tarefa
FOR EACH ROW
BEGIN
    IF OLD.status <> NEW.status THEN
        INSERT INTO historico_tarefa
        (id_tarefa, tipo_evento, status_anterior,
         status_novo, descricao)
        VALUES
        (NEW.id_tarefa, 'ALTERACAO_STATUS',
         OLD.status, NEW.status,
         'Status da tarefa alterado');
    END IF;
END
""")

conexao.commit()

print("Trigger de alteração de status criado!")

cursor.close()
conexao.close()


In [ ]:

def atribuir_tarefa(id_tarefa, id_funcionario):
    conexao = mysql.connector.connect(
        host="127.0.0.1",
        user="hitech_user",
        password="Hitech@2026",
        database="hitech_erp"
    )

    cursor = conexao.cursor()

    try:
        cursor.execute("""
            INSERT INTO alocacao_tarefa
            (id_tarefa, id_funcionario)
            VALUES (%s, %s)
        """, (id_tarefa, id_funcionario))

        cursor.execute("""
            INSERT INTO historico_tarefa
            (id_tarefa, id_funcionario, tipo_evento, descricao)
            VALUES (%s, %s, 'ATRIBUICAO',
                    'Tarefa atribuida a funcionario')
        """, (id_tarefa, id_funcionario))

        conexao.commit()
        print("Tarefa atribuída com sucesso!")

    except mysql.connector.Error as erro:
        conexao.rollback()
        print("Não foi possível atribuir a tarefa:", erro)

    finally:
        cursor.close()
        conexao.close()


exemplo de ultilização:

In [ ]:

atribuir_tarefa(1, 1)
atribuir_tarefa(2, 3)
atribuir_tarefa(3, 5)


# operações crud

In [ ]:

conexao = mysql.connector.connect(
    host="127.0.0.1",
    user="hitech_user",
    password="Hitech@2026",
    database="hitech_erp"
)

cursor = conexao.cursor()

cursor.execute("""
SELECT
    p.id_projeto,
    p.nome,
    c.nome AS cliente,
    p.status
FROM projeto p
JOIN cliente c ON c.id_cliente = p.id_cliente
ORDER BY p.id_projeto
""")

for linha in cursor.fetchall():
    print(linha)

cursor.close()
conexao.close()


In [ ]:

def atualizar_status(id_tarefa, novo_status):
    conexao = mysql.connector.connect(
        host="127.0.0.1",
        user="hitech_user",
        password="Hitech@2026",
        database="hitech_erp"
    )

    cursor = conexao.cursor()

    cursor.execute("""
        UPDATE tarefa
        SET status = %s
        WHERE id_tarefa = %s
    """, (novo_status, id_tarefa))

    conexao.commit()

    print("Status atualizado:", cursor.rowcount)

    cursor.close()
    conexao.close()

atualizar_status(1, "Em andamento")


In [ ]:

def finalizar_alocacao(id_alocacao):
    conexao = mysql.connector.connect(
        host="127.0.0.1",
        user="hitech_user",
        password="Hitech@2026",
        database="hitech_erp"
    )

    cursor = conexao.cursor()

    cursor.execute("""
        UPDATE alocacao_tarefa
        SET ativa = FALSE,
            data_fim = CURRENT_TIMESTAMP
        WHERE id_alocacao = %s
          AND ativa = TRUE
    """, (id_alocacao,))

    conexao.commit()

    print("Alocação finalizada:", cursor.rowcount)

    cursor.close()
    conexao.close()


In [ ]:

def excluir_tarefa(id_tarefa):
    conexao = mysql.connector.connect(
        host="127.0.0.1",
        user="hitech_user",
        password="Hitech@2026",
        database="hitech_erp"
    )

    cursor = conexao.cursor()

    try:
        cursor.execute(
            "DELETE FROM tarefa WHERE id_tarefa = %s",
            (id_tarefa,)
        )
        conexao.commit()
        print("Tarefa excluída:", cursor.rowcount)

    except mysql.connector.Error as erro:
        conexao.rollback()
        print("Não foi possível excluir:", erro)

    finally:
        cursor.close()
        conexao.close()


In [ ]:

conexao = mysql.connector.connect(
    host="127.0.0.1",
    user="hitech_user",
    password="Hitech@2026",
    database="hitech_erp"
)

cursor = conexao.cursor()

cursor.execute("""
SELECT status, COUNT(*) AS quantidade
FROM tarefa
GROUP BY status
ORDER BY quantidade DESC
""")

print("TAREFAS POR STATUS")

for linha in cursor.fetchall():
    print(linha)

cursor.close()
conexao.close()


In [ ]:

conexao = mysql.connector.connect(
    host="127.0.0.1",
    user="hitech_user",
    password="Hitech@2026",
    database="hitech_erp"
)

cursor = conexao.cursor()

cursor.execute("""
SELECT
    f.nome,
    np.nome_nivel,
    t.titulo,
    t.complexidade
FROM alocacao_tarefa a
JOIN funcionario f ON f.id_funcionario = a.id_funcionario
JOIN nivel_profissional np ON np.id_nivel = f.id_nivel
JOIN tarefa t ON t.id_tarefa = a.id_tarefa
WHERE a.ativa = TRUE
ORDER BY f.nome
""")

for linha in cursor.fetchall():
    print(linha)

cursor.close()
conexao.close()


In [ ]:

conexao = mysql.connector.connect(
    host="127.0.0.1",
    user="hitech_user",
    password="Hitech@2026",
    database="hitech_erp"
)

cursor = conexao.cursor()

cursor.execute("""
SELECT
    h.id_historico,
    t.titulo,
    h.tipo_evento,
    h.status_anterior,
    h.status_novo,
    h.data_evento
FROM historico_tarefa h
JOIN tarefa t ON t.id_tarefa = h.id_tarefa
ORDER BY h.data_evento DESC
""")

for linha in cursor.fetchall():
    print(linha)

cursor.close()
conexao.close()


# Interface visual

In [ ]:

!pip install -q streamlit


In [ ]:

import streamlit as st
import mysql.connector

st.set_page_config(
    page_title="Hitech ERP",
    page_icon="💻",
    layout="wide"
)

st.title("Hitech ERP")
st.subheader("Sistema de Gestão de Projetos e Tarefas")

menu = st.sidebar.selectbox(
    "Menu principal",
    ["Painel", "Clientes", "Funcionários", "Projetos", "Tarefas"]
)

def conectar():
    return mysql.connector.connect(
        host="127.0.0.1",
        user="hitech_user",
        password="Hitech@2026",
        database="hitech_erp"
    )

if menu == "Painel":
    st.header("Painel geral")

    conexao = conectar()
    cursor = conexao.cursor()

    cursor.execute("SELECT COUNT(*) FROM projeto")
    projetos = cursor.fetchone()[0]

    cursor.execute("SELECT COUNT(*) FROM tarefa")
    tarefas = cursor.fetchone()[0]

    cursor.execute("SELECT COUNT(*) FROM funcionario")
    funcionarios = cursor.fetchone()[0]

    col1, col2, col3 = st.columns(3)

    col1.metric("Projetos", projetos)
    col2.metric("Tarefas", tarefas)
    col3.metric("Funcionários", funcionarios)

    cursor.close()
    conexao.close()

elif menu == "Clientes":
    st.header("Clientes")

    conexao = conectar()
    cursor = conexao.cursor(dictionary=True)

    cursor.execute("SELECT * FROM cliente")
    st.dataframe(cursor.fetchall())

    cursor.close()
    conexao.close()

elif menu == "Funcionários":
    st.header("Funcionários")

    conexao = conectar()
    cursor = conexao.cursor(dictionary=True)

    cursor.execute("""
        SELECT f.nome, f.email, n.nome_nivel
        FROM funcionario f
        JOIN nivel_profissional n
        ON n.id_nivel = f.id_nivel
    """)

    st.dataframe(cursor.fetchall())

    cursor.close()
    conexao.close()

elif menu == "Projetos":
    st.header("Projetos")

    conexao = conectar()
    cursor = conexao.cursor(dictionary=True)

    cursor.execute("""
        SELECT p.nome, c.nome AS cliente, p.status
        FROM projeto p
        JOIN cliente c ON c.id_cliente = p.id_cliente
    """)

    st.dataframe(cursor.fetchall())

    cursor.close()
    conexao.close()

elif menu == "Tarefas":
    st.header("Tarefas")

    conexao = conectar()
    cursor = conexao.cursor(dictionary=True)

    cursor.execute("""
        SELECT id_tarefa, titulo, complexidade, status
        FROM tarefa
    """)

    st.dataframe(cursor.fetchall())

    cursor.close()
    conexao.close()


teste de funcionamento

In [ ]:

conexao = mysql.connector.connect(
    host="127.0.0.1",
    user="hitech_user",
    password="Hitech@2026",
    database="hitech_erp"
)

cursor = conexao.cursor()

tabelas = [
    "cliente",
    "nivel_profissional",
    "funcionario",
    "projeto",
    "tarefa",
    "alocacao_tarefa",
    "historico_tarefa"
]

print("RELATÓRIO DE TESTES HITECH")
print("==========================")

for tabela in tabelas:
    cursor.execute(f"SELECT COUNT(*) FROM `{tabela}`")
    quantidade = cursor.fetchone()[0]
    print(f"{tabela}: {quantidade} registros")

print("\nVerificação concluída!")

cursor.close()
conexao.close()
